# Initialize DataXID and Generate Your First Synthetic Data

In [1]:
!pip install dataxid


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import dataxid
import os
import pandas as pd
import requests
from sklearn.datasets import fetch_openml

In [4]:
dataxid.api_key = os.getenv("DATAXID_API_KEY")

In [5]:
# read an existing dataset (adult income data)
def load_data(n_rows: int = 500) -> pd.DataFrame:
    dataset = fetch_openml(name="adult", version=2, as_frame=True, parser="auto")
    df = dataset.frame.head(n_rows).reset_index(drop=True)
    df = df.drop(columns=["fnlwgt", "education-num"], errors="ignore")
    return df

In [6]:
# take only 500 rows from the dataset (adult income data)
df = load_data(n_rows=500)
print(f"Dataset: {len(df)} rows x {len(df.columns)} cols — {list(df.columns)}")
df

Dataset: 500 rows x 13 cols — ['age', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'class']


,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,Some-college,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,55,Private,Some-college,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,36,United-States,<=50K
496,19,Private,Some-college,Never-married,Machine-op-inspct,Own-child,White,Male,0,0,40,United-States,<=50K
497,35,Private,HS-grad,Never-married,Exec-managerial,Not-in-family,White,Male,0,0,60,United-States,<=50K
498,34,Private,5th-6th,Never-married,Machine-op-inspct,Other-relative,Asian-Pac-Islander,Male,0,0,40,Laos,<=50K


In [7]:
# generate 100 synthetic data using dataxid synthesize function
synthetic = dataxid.synthesize(data=df, n_samples=100)

In [8]:
# see the generated data
synthetic

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,33,Private,Bachelors,Married-civ-spouse,Sales,Husband,Black,Male,0,0,55,United-States,<=50K
1,21,Private,HS-grad,Married-civ-spouse,Sales,Husband,White,Male,0,0,40,United-States,<=50K
2,27,Private,Some-college,Never-married,Machine-op-inspct,Not-in-family,White,Female,0,0,35,United-States,<=50K
3,22,<NA>,7th-8th,Married-civ-spouse,<NA>,Husband,White,Male,0,0,10,United-States,<=50K
4,44,Private,HS-grad,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,58,Private,10th,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K
96,41,Private,Some-college,Never-married,Other-service,Own-child,White,Female,0,0,32,United-States,<=50K
97,21,Private,Bachelors,Never-married,Exec-managerial,Not-in-family,White,Male,0,0,40,United-States,<=50K
98,18,Private,Some-college,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K


# Use LLMs to Test DataXID SDK

In [9]:
!pip install ollama


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import ollama
import json

In [11]:
from ollama import chat

response = chat(
    model='llama3.1:8b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [81]:
# create an agent persona

AGENT_PERSONAS = {
   "Functional Specialist Goblin": """
    Role: Senior SDET & Functional Testing Specialist.
    Objective: Validate DataXID SDK correctness and reliability under production-like conditions. Ensure the SDK's output is not just "working," but mathematically and logically sound.

    Exploration & Consistency Mandate:
    - Validation Mode: You must alternate between "Functional Domains" in each run (e.g., Run 1: Determinism, Run 2: Distribution Preservation, Run 3: Parameter Sensitivity).
    - Entropy Level: Medium-High. Avoid testing only "Happy Path" basics. If a previous run validated simple training, the next MUST validate a complex cross-field relationship or a specific privacy-utility trade-off.

    Functional Validation Strategy (Targeted Subsets):
    1. Determinism & Seed Integrity: Combine [seed, privacy_enabled, n_samples] to ensure that identical seeds produce identical results (Idempotency).
    2. Distribution Integrity: Combine [encoding_types, model_size, embedding_dim] to verify that the output data maintains the same statistical properties as the input.
    3. Convergence Logic: Combine [max_epochs, early_stop_patience, val_split] to confirm the SDK stops training at the logical peak.
    4. Privacy Sensitivity: Combine [privacy_enabled: True, privacy_noise] to verify that higher noise correctly (and predictably) degrades performance or changes output variance.

    Diversity & Anti-Lazy Rules:
    1. Forbidden Failure: Do NOT target empty 'data_dict' or obvious TypeErrors. These are already covered. 
    2. Data Scenario Rotation: Each run must use a different data domain:
       - Run A: Financial records (decimals, timestamps).
       - Run B: E-commerce user behavior (categorical, boolean).
       - Run C: Sensor telemetry (high-precision floats, noisy signals).
    3. Mandatory Column Mix: Your 'data_dict' must include at least one of each: TABULAR_CATEGORICAL, TABULAR_NUMERIC_AUTO, and TABULAR_DATETIME. Example:
        encoding_types={
                "zip_code": "TABULAR_CATEGORICAL",     # treat as category, not number
                "event_date": "TABULAR_DATETIME",       # force datetime encoding
                "age": "TABULAR_NUMERIC_AUTO"	# continuous numeric values
            })

    Execution Strategy:
    - If you are testing 'Determinism', you MUST provide a complex dataset that would normally be hard to reproduce.
    - If you are testing 'Privacy', ensure your data has enough variance to actually be affected by noise.

    Execution Rules:
    - High-Signal Scenarios: Success is when you design a test that proves a specific SDK feature (e.g., Early Stopping) works exactly as documented.
    - Zero Triviality: Do not test "will it run." Test "will it perform." If the SDK produces a correct output but violates a statistical property (e.g., mean shift), the test is a "High-Quality Find."
    - Strict Data Quality: Use production-grade, well-formed datasets. No adversarial poisoning here; focus on logical correctness.
    CRITICAL ANTI-HALLUCINATION RULE:
        You are STRICTLY FORBIDDEN from inventing parameter names or making assumptions about the SDK API. 
        - You must ONLY use the exact keys listed in the "Functional Validation" section.
        - Do NOT use camelCase (e.g., use 'learning_rate', not 'learningRate').
        - Do NOT use synonyms (e.g., use 'max_epochs', not 'epochs'.).
        - Any JSON output containing unauthorized keys will cause a catastrophic system failure.
        - ALL JSON keys in "params" and "config_params" MUST be in lowercase.
        - FORBIDDEN: NEVER use UPPERCASE keys (e.g., use 'seed', NOT 'SEED').
        - Only the values for 'encoding_types' should be UPPERCASE (e.g., "TABULAR_DATETIME"). All other parameter NAMES must be lowercase.
        - If you violate the case-sensitivity rule, the test will be invalid.
        - MODEL SIZE are only small, medium or large.
        
    Required JSON Output:
    {
      "validation_focus": "Specific functional property (e.g., 'Verification of Seed-based Idempotency')",
      "reasoning": "Technical explanation of the expected logical/statistical behavior and how this test proves it.",
      "category": "Category 1: Functional",
      "functionality": "Label (e.g., 'Deterministic Convergence')",
      "data_dict": {"col1": [values], "col2": [values]},
      "params": {
          "n_samples": (int),
          "config_params": {
              "seed": 42,
              ... (Insert strictly 2-4 highly relevant parameters from the mandated subsets here.)
          }
      }
    }
    """,
    "Developer Experience Critic Elf": """
    Role: Senior DX Engineer & SDK Specialist.
    Objective: Identify friction, ambiguity, and poor ergonomics in the DataXID SDK. Target "Intuition-Reality" gaps where the SDK behaves correctly internally but misleads, frustrates, or confuses the developer.

    Exploration & Entropy Mandate:
    - Exploration Mode: You must alternate between different "DX Friction Vectors" in each run (e.g., Run 1: Ambiguity, Run 2: Silent Ignoring, Run 3: Debuggability).
    - Entropy Level: High. Avoid repetitive 'Naming Collision' or basic typing critiques. If you found a DX issue in a previous iteration, you MUST move to a completely different friction vector.

    DX Friction Tactics:
    1. Ambiguity & Traps: Overloaded column meanings, case-sensitivity traps, or inputs that feel valid intuitively but aren't (e.g., passing a list of ints instead of strings for categories).
    2. Silent Overrides: Parameters (like seed, timeout, or dropout) that are silently dropped, ignored, or overridden by defaults without logging a visible warning.
    3. The "Near-Valid" Misuse: Realistically incorrect inputs based on auto-complete reliance (e.g., assuming time_limit_seconds takes an int, or mixing up batch_size and accumulation_steps).
    4. Poor Debuggability: Inducing deeply nested, vague, or non-actionable stack traces (e.g., triggering a raw NumPy/PyTorch error deep in the stack rather than a clear SDK validation error).

    Execution Rules:
    - Prioritize "Wasted Time": Success is when the SDK accepts a bad config and wastes time doing the wrong thing, OR crashes with a highly confusing, low-level internal error.
    - Zero Redundancy (The Quality Bar): If the developer's mistake results in a clear, helpful validation error (e.g., "ValueError: val_split must be between 0.1 and 0.9"), your test FAILED. The SDK did its job well. Increase 'Entropy' and try a more subtle misuse.
    - Assume the developer relies on intuition and IDE auto-complete over deep documentation reading.
    CRITICAL ANTI-HALLUCINATION RULE:
        You are STRICTLY FORBIDDEN from inventing parameter names or making assumptions about the SDK API. 
        - You must ONLY use the exact keys listed in the "SDK Parameters" section.
        - Do NOT use camelCase (e.g., use 'learning_rate', not 'learningRate').
        - Do NOT use synonyms (e.g., use 'max_epochs', not 'epochs'.).
        - Any JSON output containing unauthorized keys will cause a catastrophic system failure.

    SDK Parameters:
    - encoding_types: (dict),
    - val_split: (float),
    - batch_size: (int),
    - model_size: ("small"|"medium"|"large"),
    - embedding_dim: (int)
    
    Required JSON Output:
    {
      "dx_focus": "Current DX vector being targeted (e.g., 'Silent Ignore of learning_rate when using default optimizer')",
      "reasoning": "Explain the specific friction point, why a developer would naturally make this mistake, and the mental model mismatch.",
      "category": "Category 3: Developer Experience",
      "dx_issue": "Label (e.g., 'Silent Parameter Ignore' or 'Unhelpful Stack Trace')",
      "data_dict": {"col1": [values], "col2": [values]},
      "params": {
          "n_samples": (int),
          "config_params": {
              "encoding_types": (dict),
              "val_split": (float),
              "batch_size": (int),
              "model_size": ("small"|"medium"|"large"),
              "embedding_dim": (int)
              ... (Select only relevant parameters that contribute to the dx_focus.)
          }
      }
    }
    """,
    "Edge-Case Junior": """
    Role: Senior SDET & Chaos Engineer.
    Objective: Generate high-precision adversarial test cases for the DataXID SDK. Target "Semantic Corruption" where results are produced but are mathematically or logically invalid.

    Exploration & Entropy Mandate:
    - Exploration Mode: You must alternate between different "Attack Vectors" in each run (e.g., Run 1: Numerical, Run 2: Temporal, Run 3: Schema/Type).
    - Entropy Level: High. Avoid repetitive 'AttributeError' or 'List' type mismatches. If you found a bug in a previous iteration, you MUST move to a completely different SDK subsystem.

    Adversarial Tactics:
    1. Parameter-Data Inconsistency: Valid syntax but impossible semantics (e.g., batch_size > n_samples).
    2. Structural Poisoning: Mixed dtypes or ragged arrays bypassing initial checks.
    3. Numerical Instability: Forcing NaN/Infs or gradient issues via extreme cardinality vs small models.
    4. Silent Skew: Future-dated records in training (temporal leakage).

    Execution Rules:
    - Prioritize High-Quality Fails: Success is when the code completes (exit 0) but the output is silent garbage (skewed distributions, leaked data, corrupted embeddings).
    - Diverse Subsystems: Actively rotate focus between Preprocessing, Privacy (Noise Injection), Model Architecture, and Evaluation.
    - Zero Redundancy: If a test case results in a simple Python Exception, it is a failed attempt. Increase 'Entropy' and target deeper logic.
    CRITICAL ANTI-HALLUCINATION RULE:
        You are STRICTLY FORBIDDEN from inventing parameter names or making assumptions about the SDK API. 
        - You must ONLY use the exact keys listed in the "SDK Parameters" section.
        - Do NOT use camelCase (e.g., use 'learning_rate', not 'learningRate').
        - Do NOT use synonyms (e.g., use 'max_epochs', not 'epochs'.).
        - Any JSON output containing unauthorized keys will cause a catastrophic system failure.
        - Encoding Types are only TABULAR_CATEGORICAL, TABULAR_DATETIME, TABULAR_NUMERIC_AUTO.

    SDK Parameters:
    - embedding_dim: (int),
    - val_split: (float),
    - learning_rate: (float | None),
    - encoding_types: (dict) 

    
    Required JSON Output:
    {
      "exploration_focus": "Current subsystem being targeted (e.g., 'Privacy-driven NaN cascade')",
      "reasoning": "Technical explanation of the intended silent failure.",
      "category": "Adversarial Edge Case",
      "failure_mode": "Label",
      "data_dict": {"col1": [values]},
      "params": {
          "n_samples": (int),
          "config_params": {
              "embedding_dim": (int),
              "val_split": (float),
              "learning_rate": (float | None),
              "encoding_types": (dict)
          }
      }
    }
    """
}

In [78]:
def run_agentic_test(persona_name, iteration_count=3):
    persona_prompt = AGENT_PERSONAS.get(persona_name)
    agent_history = []
    for i in range(iteration_count):
        print(f"\n--- 🤖 {persona_name.upper()} EXPERIMENT #{i+1} STARTING ---")

        past_errors = [
            f"Iter {h['iteration']}: {h.get('error', 'No Error')}" 
            for h in agent_history if h['status'] == "BUG FOUND"
        ]
        
        history_context = ""
        if past_errors:
            history_context = f"AVOID REPEATING THESE PAST ERRORS:\n" + "\n".join(past_errors) + "\nTarget a completely different feature or logic branch."

        # 1. Ask the Brain for a new scenario
        user_message = f"{history_context}\nGenerate a new, unique, and challenging scenario for {persona_name}. Ensure strictly valid JSON format."

        response = ollama.chat(
            model='llama3.1:8b',
            messages=[
                {"role": "system", "content": persona_prompt},
                {"role": "user", "content": user_message}
            ],
            format='json' 
        )

        content = response['message']['content']
        cleaned_content = content.replace("```json", "").replace("```", "").strip()

        # If there is JSON error then skip 
        try:
            test_case = json.loads(cleaned_content)
        except:
            print(f"⚠️ Agent failed to generate valid JSON. Skipping iteration. Error: {str(e)}")
            continue

            print(f"🧐 Reasoning: {test_case.get('reasoning', 'No reasoning provided')}")

        # If agent model try to select most simple way which is empty dict then skip it 
        if not test_case.get('data_dict') or not test_case.get('params'):
            print("⚠️ Agent generated an empty data_dict or params. Skipping trivial test.")
            continue
            
        try:
            # 2. Prepare Data and Config
            df = pd.DataFrame(test_case['data_dict'])
            if df.empty:
                print("⚠️ Generated DataFrame is empty. Skipping.")
                continue
                
            config_params = test_case['params'].get('config_params', {})
            config = dataxid.ModelConfig(**config_params)
            n_samples = test_case['params'].get('n_samples', max(1, len(df)))

        except Exception as data_prepration_e:
            print(f"⚠️ Pre-execution Setup Error (Bad config params provided by LLM): {str(data_prepration_e)}")
            continue

        try:
            # 3. Execute SDK Method
            print(f"🚀 Testing dataxid.synthesize() with {len(df)} rows...")
            result = dataxid.synthesize(data=df, n_samples=n_samples, config=config)
            
            # Save success with full context
            agent_history.append({
                "iteration": i + 1,
                "status": "SUCCESS",
                "reasoning": test_case.get('reasoning'),
                "input_data": test_case.get('data_dict'),
                "input_params": test_case.get('params')
            })
            print("✅ SDK passed the test.")
        except Exception as e:
            # 4. Bug Analysis & Formal Report Generation
            error_msg = str(e)
            print(f"🔥 BUG FOUND: {error_msg}")

            # Safely extract context variables so the LLM can write the "Steps to Reproduce"
            current_reasoning = test_case.get('reasoning', 'N/A') if 'test_case' in locals() else 'N/A'
            current_data = test_case.get('data_dict', {}) if 'test_case' in locals() else {}
            current_params = test_case.get('params', {}) if 'test_case' in locals() else {}

            # Construct a prompt with a strict structural template
            analysis_prompt = f"""
            You are a Senior QA Lead. The DataXID SDK just crashed during an adversarial test.
            
            Context of the crash:
            - Error Message: {error_msg}
            - Adversarial Reasoning: {current_reasoning}
            - Input Data: {current_data}
            - Input Parameters: {current_params}
            
            Based on standard bug tracking rules, analyze the severity and generate a formal Bug Report. 
            Do not include conversational filler. Return ONLY the report in the exact Markdown format below:
            
            **Severity Score:** [e.g., CRITICAL / HIGH / MEDIUM / LOW] - [One sentence justification]
            
            ### 🐛 Bug Report
            **Title:** [Clear, concise technical title summarizing the crash]
            
            **Description:** [Brief explanation of what happened, why it is a failure, and the underlying mechanism that broke]
            
            **Steps to Reproduce:**
            1. Initialize SDK configuration with: `config_params = ...`
            2. Prepare dataset with structure: `data_dict = ...`
            3. Execute process with `n_samples = ...`
            4. Observe the crash.
            
            **Expected Behavior:** [How the SDK *should* have handled this edge-case gracefully without crashing]
            
            **Actual Behavior:** Crashes with exception: `{error_msg}`
            """
            
            analysis_response = ollama.chat(
                model='llama3.1:8b',
                messages=[{"role": "user", "content": analysis_prompt}]
            )

            bug_report = analysis_response['message']['content']
            print(f"\n📊 Generated Bug Report:\n{bug_report}\n" + "-"*50)

            # Save bug with full context and the generated markdown report
            agent_history.append({
                "iteration": i + 1,
                "status": "BUG FOUND",
                "error": error_msg,
                "bug_report_markdown": bug_report,
                "reasoning": current_reasoning,
                "input_data": current_data,
                "input_params": current_params
            })

    return agent_history

In [86]:
# Start the Edge-Case Junior
results_ec = run_agentic_test("Edge-Case Junior", 5)


--- 🤖 EDGE-CASE JUNIOR EXPERIMENT #1 STARTING ---
🚀 Testing dataxid.synthesize() with 3 rows...
✅ SDK passed the test.

--- 🤖 EDGE-CASE JUNIOR EXPERIMENT #2 STARTING ---
🚀 Testing dataxid.synthesize() with 3 rows...
✅ SDK passed the test.

--- 🤖 EDGE-CASE JUNIOR EXPERIMENT #3 STARTING ---
🚀 Testing dataxid.synthesize() with 3 rows...
✅ SDK passed the test.

--- 🤖 EDGE-CASE JUNIOR EXPERIMENT #4 STARTING ---
🚀 Testing dataxid.synthesize() with 2 rows...
🔥 BUG FOUND: unhashable type: 'list'

📊 Generated Bug Report:
**Severity Score:** CRITICAL - The DataXID SDK crashed unexpectedly due to an unhandled error condition, indicating a potential instability or data corruption issue.

### 🐛 Bug Report
**Title:** Crash on Adversarial Input with Large Batch Size and Unhashable Type Error

**Description:** During an adversarial test, the DataXID SDK crashed when processing input data containing extremely large feature values. The error was triggered by attempting to hash a list object as a key in

KeyboardInterrupt: 

In [95]:
# Start the Functional Specialist
results_fs = run_agentic_test("Functional Specialist Goblin", 3)


--- 🤖 FUNCTIONAL SPECIALIST GOBLIN EXPERIMENT #1 STARTING ---
⚠️ Pre-execution Setup Error (Bad config params provided by LLM): ModelConfig.__init__() got an unexpected keyword argument 'noise_stddev'

--- 🤖 FUNCTIONAL SPECIALIST GOBLIN EXPERIMENT #2 STARTING ---
🚀 Testing dataxid.synthesize() with 3 rows...
🔥 BUG FOUND: only integer tensors of a single element can be converted to an index

📊 Generated Bug Report:
**Severity Score:** HIGH - The SDK's failure to handle high-dimensional sensor data under increasing levels of privacy noise results in a crash, indicating potential loss of critical functionality.

### 🐛 Bug Report
**Title:** DataXID SDK Crashes When Processing High-Dimensional Sensor Data with Privacy Noise

**Description:** During an adversarial test, the DataXID SDK crashed when processing high-dimensional sensor data with increasing levels of privacy noise. This is a failure because the SDK is expected to maintain performance and handle edge-cases without crashing.

**S

KeyboardInterrupt: 

In [75]:
results_dx = run_agentic_test("Developer Experience Critic Elf", 3)


--- 🤖 DEVELOPER EXPERIENCE CRITIC ELF EXPERIMENT #1 STARTING ---
⚠️ Pre-execution Setup Error (Bad config params provided by LLM): ModelConfig.__init__() got an unexpected keyword argument '# Hidden default cap is 512 here, but SDK won't warn or error on this value (yet...). This value will silently be capped at 512 internally and performance will suffer as a result. Not the worst DX issue, but it's 'good' enough for this Critic Elf run :). Don't get too comfortable though, we're only getting started..) # noqa: E501'

--- 🤖 DEVELOPER EXPERIENCE CRITIC ELF EXPERIMENT #2 STARTING ---
🚀 Testing dataxid.synthesize() with 2 rows...
🔥 BUG FOUND: empty() received an invalid combination of arguments - got (tuple, dtype=NoneType, device=torch.device), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)

In [96]:
# Save results to a JSON file locally
results_to_save = {
    "dataxid_edge_case_report.json": results_ec,
    "dataxid_functional_specialist_report.json": results_fs,
    "dataxid_dx_report.json": results_dx
}

for filename, data in results_to_save.items():
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"✅ All test results have been saved locally to: {filename}")

✅ All test results have been saved locally to: dataxid_edge_case_report.json
✅ All test results have been saved locally to: dataxid_functional_specialist_report.json
✅ All test results have been saved locally to: dataxid_dx_report.json


# Bir takım notlar
Agentlar öyle bir şekilde yönlendirilmeli ki önceki yakaladığı hatayı yine yakalamaya çalışmak yerine farklı yerlere yönelsin bu yüzden de Exploration ve Entropy adında iki değer atadık. 
Ayrıca agent çıktı vermeden önce ona odaklanabileceği bir yön biçiyoruz bu şekilde daha spesifik hale geliyor. Konfigürasyon parametrelerini de her agentın yaptığı işe göre atamak önemli
gereksiz parametreler agentı yolundan saptırıyor. Bunu şu şekilde de yapabilirdik: parametleri bir havuzda toplardık ve agenta buradan sadece ilgili olan parametreleri seçmesini söylerdik.
"Statistical Integrity" (İstatistiksel Tutarlılık) ve "Behavioral Regression" (Davranışsal Regresyon)
Sistem bazı hataları SDK hatası olarak döndürüyor ancak bunlar LLM'in kendi hataları:
*JSON Ayrıştıma(Parse) hatası
*Context Window Zehirlenmesi -> geçmişteki tüm hataları veri setleri(data_dict) modele verince modeiln context limiti dolabilir ve halüsinasyon görmeye başlayabiliyor. Bu yüzden önceki tüm hataları vermek yerine sadece hemen önceki hatayı modele verirsek daha iyi olur.
*Global Test Geçmişi -> test geçmişini globala tanımladığımdan tüm agentlar öncekinin hatlarınıda görüyor, önceki agentların hata raporlarını. Bu istemediğimiz bir şey, agent sadece kendi işiyle uğraşacak.
*Data_dict boş olduğunda model bu yolu seçmeye başlıyor ve bunun sonucunda verdiği tüm bug çıktıları data_dict boş diye veriyor.
synthesize() işlemi çok uzun sürdüğünden ThreadPoolExecutor kullanmaya karar verdim bu şekilde işlemi kesebileceğimi zannetmiştim ancak ana program error fırlatmasına rağmen dataxid.synthesize işi durmuyor haliyle Python bunun bitmesini bekliyor. 
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(dataxid.synthesize, data=df, n_samples=n_samples, config=config)

                # if future.result() exceeds the MAX_WAIT_SECONDS then throw TimeoutError
                result = future.result(timeout=MAX_WAIT_SECONDS)

Thread yerine multiprocessing kullanmaya karar verdim. Process ile karmaşıklaşmaya başladığı için sildim.
